In [126]:
#imports
from pyspark.sql.functions import *
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, StringType, TimestampType, BooleanType, BinaryType
from datetime import datetime
import ConnectionConfig as cc
debugging_mode=True


In [127]:
#config
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_USER",4)
spark.getActiveSession()

run_timestamp = datetime.now()

Environment variables are set...


In [128]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [129]:
#EXTRACT

# user tabel
df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "user_table") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("users")

# treusure log tabel
df_treasure_log = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_log") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure_log.createOrReplaceTempView("treasure_log")

#treusure tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("treasure")

#city tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "city") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("city")

#country tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "country") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("country")

INITIAL

In [130]:
#TRANSFORM

#base user
base_user = spark.sql("""
SELECT
        u.id AS userId,
        u.first_name AS first_Name,
        u.last_name AS last_Name,
        u.mail AS email,
        u.city_city_id as cityId,
        CONCAT(u.street, ' ', u.number) AS address,
        COALESCE(tl.treasure_count, 0) AS treasure_count
    FROM users u
    LEFT JOIN (
        SELECT hunter_id, COUNT(*) AS treasure_count
        FROM treasure_log
        GROUP BY hunter_id
    ) tl ON u.id = tl.hunter_id
""")

base_user.createOrReplaceTempView("baseUser")

In [137]:
#TRANSFORM

#expierenceLevel berekenen
expierencelevel_user = spark.sql("""
SELECT
        userId,
        CASE
            WHEN treasure_count = 0 THEN 'Starter'
            WHEN treasure_count < 4 THEN 'Amateur'
            WHEN treasure_count BETWEEN 4 AND 10 THEN 'Professional'
            ELSE 'Pirate'
        END AS experienceLevel
    FROM baseUser
""")
expierencelevel_user.createOrReplaceTempView("expierenceLevelUser")

+--------------------+---------------+
|              userId|experienceLevel|
+--------------------+---------------+
|[0A 12 E8 9C 28 E...|        Starter|
|[0D 54 38 BE 9C F...|        Starter|
|[18 A7 A0 C4 D1 F...|        Starter|
|[1E 12 A8 78 3B D...|        Starter|
|[26 A5 19 60 DE 7...|        Starter|
|[2C 6D 17 E6 AE 2...|        Starter|
|[43 EE F3 DB 64 5...|        Starter|
|[4E 4F 1D 31 A9 D...|        Starter|
|[51 C8 52 A0 FC D...|        Starter|
|[58 8A 3B 83 4E A...|        Starter|
|[60 0F 18 78 03 6...|        Starter|
|[64 63 7E 47 B6 D...|        Starter|
|[6E DF E4 DF 60 3...|        Starter|
|[70 B6 5E 36 94 C...|        Starter|
|[71 66 B6 48 01 7...|        Starter|
|[74 CB 16 1C CC 4...|        Starter|
|[7B 57 01 E3 BF A...|        Starter|
|[7F 0F BA 1F AE F...|        Starter|
|[80 78 2C E7 51 D...|        Starter|
|[81 23 51 9E 40 F...|        Starter|
+--------------------+---------------+
only showing top 20 rows


In [132]:
#TRANSFORM

#dedicator berekenen
dedicator_user = spark.sql("""
   SELECT
        u.userId,
        CASE WHEN COUNT(t.id) > 0 THEN TRUE ELSE FALSE END AS dedicator
    FROM baseUser u
    LEFT JOIN treasure t ON u.userId = t.owner_id
    GROUP BY u.userId
""")
dedicator_user.createOrReplaceTempView("dedicatorUser")

In [133]:
#TRANSFORM

#Country berekenen
country_user = spark.sql("""
    SELECT
        u.userId,
        co.code as country
    FROM baseUser u
    JOIN city ct on ct.city_id = u.cityId
    join country co  on co.code = ct.country_code
""")

country_user.createOrReplaceTempView("countryUser")

In [134]:
#TRANSFORM

#zet alles samen
complete_user = spark.sql(f"""
SELECT
    b.userId,
    b.first_Name,
    b.last_Name,
    b.email,
    b.address,
    e.experienceLevel,
    d.dedicator,
    c.country,
    to_timestamp('{run_timestamp}') as scd_start,
    to_timestamp(null) AS scd_end,
    TRUE AS current,
    md5(CONCAT(e.experienceLevel,d.dedicator,b.first_Name,b.last_Name)) AS md5
FROM baseUser b
LEFT JOIN expierenceLevelUser e ON b.userId = e.userId
LEFT JOIN dedicatorUser d ON b.userId = d.userId
LEFT JOIN countryUser c ON b.userId = c.userId
""")

In [135]:
#LOAD

#maken deltatabel
spark.sql("DROP TABLE IF EXISTS default.dimUser")

DeltaTable.create(spark) \
    .tableName("dimUser") \
    .addColumn("userSurKey", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0, 1)) \
    .addColumn("userId", BinaryType(), nullable=False) \
    .addColumn("first_name", StringType()) \
    .addColumn("last_name", StringType()) \
    .addColumn("email", StringType()) \
    .addColumn("address", StringType()) \
    .addColumn("experiencelevel", StringType()) \
    .addColumn("dedicator", BooleanType()) \
    .addColumn("country", StringType()) \
    .addColumn("scd_start", TimestampType()) \
    .addColumn("scd_end", TimestampType()) \
    .addColumn("md5", StringType()) \
    .addColumn("current", BooleanType()) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

AnalysisException: [DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION] Cannot create table ('`default`.`dimUser`'). The associated location ('file:/home/jovyan/work/Project/spark-warehouse/dimuser') is not empty and also not a Delta table.

In [124]:
#LOAD

#write to tabel
complete_user.write.format("delta").mode("overwrite").saveAsTable("dimUser")

In [125]:
#end de spark sessie
spark.stop()